# Data Ingestion
---

## Optical: Sentinel-2
---

In [41]:
import ee
import geemap
import numpy as np
import xarray as xr
import rioxarray as rxr

In [42]:
ee.Authenticate()
ee.Initialize(
    project="multimodal-regression"
)

In [11]:
# Manually select ROI
m = geemap.Map(basemap="SATELLITE")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [43]:
# Extract bounding box
ee_roi = m.user_roi.bounds()
ee_roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Geometry.bounds",
    "arguments": {
      "geometry": {
        "functionInvocationValue": {
          "functionName": "GeometryConstructors.Polygon",
          "arguments": {
            "coordinates": {
              "constantValue": [
                [
                  [
                    -483.611193,
                    48.375325
                  ],
                  [
                    -483.611193,
                    48.490133
                  ],
                  [
                    -483.439547,
                    48.490133
                  ],
                  [
                    -483.439547,
                    48.375325
                  ],
                  [
                    -483.611193,
                    48.375325
                  ]
                ]
              ]
            },
            "geodesic": {
              "constantValue": false
            }
          }
        }
      }
    }
  }
})

In [158]:
# Fetch data

def mask_s2_clouds(image):
    mask = image.select("MSK_CLDPRB").lt(20)    # pr_cld
    return image.updateMask(mask)

def scale_img(image):
    return image.divide(10000)

# Filter collection
date_start = "2025-06-01"
date_end = "2026-09-01"
cld_percentage = 100

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_roi)
    .filterDate(date_start, date_end)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cld_percentage))
    .map(mask_s2_clouds)
    .map(scale_img)
    .select(["B2", "B3", "B4", "B8"]) # Blue, Green, Red, NIR
).sort("CLOUDY_PIXEL_PERCENTAGE",False)

# Create a median composite to collapse the time dimension for structure analysis
s2_img_med = s2_col.median().clip(ee_roi)
s2_img_cld_min = s2_col.mosaic().clip(ee_roi)

viz_experimental = ee.Image("COPERNICUS/S2_SR_HARMONIZED/20250601T191909_20250601T192647_T10UDU").clip(ee_roi).divide(10000)

# 4. Ingest directly into Xarray using wxee
print("Streaming Earth Engine image to local Xarray Dataset...")
s2_ds = geemap.ee_to_xarray(
    dataset=s2_img_med,
    geometry=ee_roi,
    scale=10,           # Sentinel-2 native resolution in meters
    crs="EPSG:4326", 
)

print("\n--- Pipeline Target Array Acquired ---")
print(s2_ds)

Streaming Earth Engine image to local Xarray Dataset...

--- Pipeline Target Array Acquired ---
<xarray.Dataset> Size: 40B
Dimensions:  (time: 1, y: 1, x: 1)
Coordinates:
  * time     (time) int64 8B 0
  * y        (y) float64 8B 45.0
  * x        (x) float64 8B -125.0
Data variables:
    B2       (time, y, x) float32 4B ...
    B3       (time, y, x) float32 4B ...
    B4       (time, y, x) float32 4B ...
    B8       (time, y, x) float32 4B ...


In [159]:
# Visualize optical data
m = geemap.Map()
true_color_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3
}

false_color_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0,
    "max": 0.3
}

qa_vis = {
    "bands": ["MSK_CLDPRB"],
}

m.centerObject(ee_roi, zoom=12)
m.addLayer(s2_img_med, true_color_vis, "True Color (RGB)")
m.addLayer(s2_img_cld_min, true_color_vis, "filtered_single")
m.addLayer(viz_experimental, true_color_vis, "unfiltered_single")
m.addLayer(viz_experimental, qa_vis, "CLDPRB)")
m

Map(center=[48.43271801983087, -123.52536999999823], controls=(WidgetControl(options=['position', 'transparent…

## SAR: Sentinel-1 C-Band
---